# Step 02 — Embedding Extraction

Extracts V-JEPA2 and DINOv2 embeddings using:
- **Top-down sub-view** from the InHARD 3-view composite
- **YOLO person crop** aligned to live inference
- **Temporal attention pooling** (not mean-pool)

Results saved to `outputs/embeddings.npz` (VJEPA) and `outputs/embeddings_dinov2.npz`.

In [ ]:
import sys
from pathlib import Path
NB_DIR = Path.cwd()
if str(NB_DIR) not in sys.path:
    sys.path.insert(0, str(NB_DIR))

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

## Preview — what gets fed to the backbones

Run this **before** full extraction to inspect:
1. Raw InHARD 3-view mosaic
2. Top-down sub-view + YOLO bounding box (green) or center-crop fallback
3. The **16 sampled person crops** actually encoded by V-JEPA / DINOv2

In [ ]:
from lib.inhard import analyze_training_clips
from lib.paths import find_inhard_root, OUTPUTS_DIR
from lib.crop_extract import visualize_crop_extraction, save_crop_debug_samples

PREVIEW_VIEW = 'topdown'
PREVIEW_N_CLASSES = 6          # one clip per class (up to this many)
PREVIEW_CLIP_INDEX = None      # set e.g. 0 to force a specific clip from the list below

root = find_inhard_root()
report = analyze_training_clips(root, clips_per_class=None)
assert report.ok, report.error

# Pick one clip per class for a quick look
seen = set()
preview_clips = []
for rec in report.clips:
    if rec.label in seen:
        continue
    seen.add(rec.label)
    preview_clips.append(rec)
    if len(preview_clips) >= PREVIEW_N_CLASSES:
        break

print(f"InHARD root: {root}")
print(f"Previewing {len(preview_clips)} clips (view={PREVIEW_VIEW})\n")
for i, rec in enumerate(preview_clips):
    print(f"  [{i}] {rec.label:28s}  {rec.path.name}")

idx = PREVIEW_CLIP_INDEX if PREVIEW_CLIP_INDEX is not None else 0
rec = preview_clips[idx]
fig = visualize_crop_extraction(
    rec.path,
    view=PREVIEW_VIEW,
    title=f"{rec.label}  ·  subject={rec.subject}  ·  {rec.path.name}",
)
plt.show()

In [ ]:
# Gallery: one PNG per class → outputs/debug_crops/
debug_dir = OUTPUTS_DIR / 'debug_crops'
paths = save_crop_debug_samples(
    report.clips, debug_dir,
    n_samples=12,
    view=PREVIEW_VIEW,
)
print(f"Saved {len(paths)} images to {debug_dir}/")
for p in paths:
    print(f"  {p.name}")

# Show all in the notebook
ncols = 2
nrows = (len(paths) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(14, 4.5 * nrows))
axes = np.atleast_1d(axes).flatten()
for ax, p in zip(axes, paths):
    ax.imshow(plt.imread(p))
    ax.set_title(p.stem.replace('__', ' · '), fontsize=9)
    ax.axis('off')
for ax in axes[len(paths):]:
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
from lib.pipeline import PipelineConfig, step_embeddings
from lib.constants import BACKBONE_VJEPA, BACKBONE_DINOV2

cfg = PipelineConfig(
    clips_per_class           = None,        # ALL clips
    inhard_view               = 'topdown',
    temporal_agg              = 'attention',
    backbones                 = (BACKBONE_VJEPA, BACKBONE_DINOV2),
    skip_embeddings_if_exists = False,       # set True to reuse cached
    viz_crop_samples          = 12,          # save outputs/debug_crops/*.png during Pass 1
)

print(f"View: {cfg.inhard_view}  |  temporal_agg: {cfg.temporal_agg}")
print("Starting embedding extraction …")
emb_results = step_embeddings(cfg)
print("Done.")

In [ ]:
# Inspect saved embeddings
from lib.paths import OUTPUTS_DIR

for backbone, info in emb_results.items():
    if info.get('skipped'):
        print(f"{backbone}: skipped (cached)")
        continue
    npz = np.load(info['path'], allow_pickle=True)
    X, y, cls = npz['X'], npz['y'], list(npz['class_names'])
    print(f"\n{backbone}:")
    print(f"  shape     : {X.shape}")
    print(f"  classes   : {len(cls)}")
    from collections import Counter
    counts = Counter(y.tolist())
    for i, name in enumerate(cls):
        bar = '█' * counts.get(i, 0)
        print(f"  {name:30s} {counts.get(i,0):4d} {bar[:50]}")

In [ ]:
# Quick PCA sanity check — are classes separable before training?
from sklearn.decomposition import PCA

for backbone, info in emb_results.items():
    if info.get('skipped'): continue
    npz = np.load(info['path'], allow_pickle=True)
    X, y, cls = npz['X'], npz['y'], list(npz['class_names'])
    X2 = PCA(n_components=2, random_state=42).fit_transform(X)

    fig, ax = plt.subplots(figsize=(9, 7))
    cmap = plt.cm.get_cmap('tab20')
    for i, name in enumerate(cls):
        mask = y == i
        ax.scatter(X2[mask,0], X2[mask,1], c=[cmap(i/max(len(cls)-1,1))],
                   label=name[:16], s=18, alpha=0.55)
    ax.set_title(f'PCA — {backbone} embeddings (before training)', fontsize=12)
    ax.legend(fontsize=7, ncol=2)
    ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
    plt.tight_layout()
    plt.savefig(OUTPUTS_DIR / f'02_pca_pretraining_{backbone}.png', dpi=130, bbox_inches='tight')
    plt.show()